# YOLO26n 새·드론 인식 모델 학습 (Google Colab)

이 노트북은 YOLOv8이 아닌 **공식 YOLO26n** 엔진을 새/드론용으로 추가 학습합니다. **아직 학습된 모델이 아니며**, 아래 4개 셀을 순서대로 실행해야 합니다.

**준비물:** 새·드론 라벨이 포함된 YOLO 형식 데이터셋 ZIP (train/valid/test 분리 권장). 데이터셋 라이선스를 확인하세요. Google Drive의 내 드라이브에 ZIP을 올리세요.

Colab 상단 **런타임 → 런타임 유형 변경 → T4 GPU**(가능한 경우) 선택. 학습에는 수십 분~수 시간이 걸릴 수 있습니다. 컴퓨터에 GPU가 없어도 Colab에서 처리됩니다.

**주의:** 이 모델은 기존 YOLO26의 **아키텍처를 유지**하지만 새·드론 2개 클래스로 전문화되므로 COCO 일반 객체 인식 80종을 그대로 보존하지 않습니다. 새 날갯짓의 시간 분석은 별도 기능입니다. 이는 민간 테스트용이며 실제 드론 탐지 성능을 보증하지 않습니다.

In [ ]:
!pip -q install "ultralytics>=8.4,<9" pyyaml
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
print("설치 완료. Google Drive에 데이터셋 ZIP이 있는지 확인하세요.")

## 2. Google Drive에 있는 데이터셋 ZIP 입력

아래 셀을 실행한 다음 ZIP 경로를 입력합니다. 예시: `/content/drive/MyDrive/bird-drone.zip`.

Roboflow에서 다운로드한 YOLO 형식 ZIP을 사용했다면 **새/드론 2개 클래스**인지 확인하세요. 같은 비행 영상의 인접 프레임은 훈련과 테스트에 동시에 넣지 마세요.

In [ ]:
import zipfile, yaml
from pathlib import Path
zip_path = Path(input("Google Drive의 YOLO 데이터셋 ZIP 전체 경로를 입력하세요: ").strip()).expanduser()
if not zip_path.is_file():
    raise FileNotFoundError(f"ZIP 파일을 찾지 못함: {zip_path}")
root = Path("/content/yolo26_bird_drone_dataset")
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        resolved = (root / member.filename).resolve()
        if not resolved.is_relative_to(root.resolve()):
            raise ValueError("압축 파일에 안전하지 않은 경로가 있습니다.")
    archive.extractall(root)
yamls = sorted(root.rglob("data.yaml"))
if not yamls:
    raise FileNotFoundError("ZIP 안에 data.yaml이 없습니다. YOLO 형식 데이터셋인지 확인하세요.")
yml = yamls[0]
metadata = yaml.safe_load(yml.read_text(encoding="utf-8"))
if not isinstance(metadata, dict):
    raise ValueError("data.yaml 구성 오류")
names = metadata.get("names", [])
if isinstance(names, dict):
    names = [names.get(i, names.get(str(i))) for i in range(len(names))]
if len(names) != 2 or {str(x).lower().strip() for x in names} != {"bird", "drone"}:
    raise ValueError(f"새/드론 2개 클래스여야 합니다: {names}")
for section, dirname in (("train", "train"), ("val", "valid"), ("test", "test")):
    location = yml.parent / dirname / "images"
    if location.is_dir():
        metadata[section] = str(location.resolve())
if not metadata.get("val"):
    raise ValueError("검증용 valid 이미지가 필요합니다.")
metadata["path"] = str(yml.parent.resolve())
fixed = yml.parent / "data_yolo26_local.yaml"
fixed.write_text(yaml.safe_dump(metadata, allow_unicode=True), encoding="utf-8")
print("데이터셋 준비 완료:", fixed, names)

## 3. YOLO26n GPU 학습 시작

데이터셋이 크면 시간이 걸립니다. 런타임이 끊기면 처음부터 다시 해야 할 수 있으므로 학습 중 종료되지 않도록 확인하세요. 학습 결과는 모델 생성 여부와 별도로 새·드론·빈 하늘 영상으로 검증해야 합니다.

In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import files
import urllib.request
script = Path("/content/train_yolo26_drone.py")
url = "https://raw.githubusercontent.com/whgnltkd123-ai/yolo-ios-app/main/tools/train_yolo26_drone.py"
urllib.request.urlretrieve(url, script)
if not os.path.exists("/usr/bin/nvidia-smi") and subprocess.run(["which","nvidia-smi"],capture_output=True).returncode != 0:
    print("경고: GPU가 없습니다. 런타임 유형을 GPU로 변경하는 것을 권장합니다.")
subprocess.run(["python", str(script), "--data", str(fixed), "--epochs", "50", "--imgsz", "512", "--batch", "8", "--device", "0", "--project", "/content/yolo26_training"], check=True)
print("학습 종료")

## 4. 가중치를 Google Drive에 저장하고 SHA256 확인

여기서 만든 `YOLO26-Bird-Drone-best.pt`가 실제 YOLO26n 새·드론 학습 모델입니다. **본인 GitHub 저장소의 Release에 올린 다음 URL을 사용**하세요. Codemagic에서 새 워크플로 `YOLO26 Drone Alert - 3 Levels`에 아래 SHA256과 다운로드 URL을 환경변수로 입력해 iOS IPA를 만들 수 있습니다. 비밀번호는 이 노트북이나 GitHub 공개 파일에 넣지 마세요.

In [ ]:
import hashlib, shutil
from pathlib import Path
out = Path("/content/yolo26_training/nano-bird-drone/weights/best.pt")
if not out.is_file():
    raise FileNotFoundError("학습된 best.pt가 없습니다. 이전 셀의 학습 로그를 확인하세요.")
dest = Path("/content/drive/MyDrive/YOLO26-Bird-Drone-best.pt")
shutil.copy2(out, dest)
digest = hashlib.sha256(dest.read_bytes()).hexdigest()
print("완성된 모델:", dest)
print("YOLO26_DRONE_CHECKPOINT_SHA256 =", digest)
print("YOLO26_DRONE_CHECKPOINT_URL = GitHub Release에 올린 best.pt 다운로드 URL")
print("실제 iPhone 성능과 새/드론 오탐·미탐은 아직 검증되지 않았습니다.")